In [1]:
import cv2
from pathlib import Path as P
from kornia.feature import LoFTR
from PIL import Image
import torchvision.transforms.functional as tvF
import torch

data_path = P('/home/dj/Downloads/project/pointrix/examples/vid_art_GS/laptop_10211')
rgb_path = data_path / 'images'
mask_path = data_path / 'masks'

rgbs = list(sorted(rgb_path.glob('*.png')))
masks = list(sorted(mask_path.glob('*.png')))

device = torch.device('cuda')

loftr = LoFTR('indoor').to(device)
save_path = data_path / 'loftr_matches'
save_path.mkdir(exist_ok=True)

In [11]:
from tqdm import tqdm
max_interval = 5
for interval in tqdm(range(max_interval)):
    for i in tqdm(range(len(rgbs))):
        
        rgb_fname = rgbs[i]
        mask_fname = masks[i]
        
        rgb = tvF.pil_to_tensor(Image.open(rgb_fname).convert('L')).unsqueeze(0).float()/255
        mask = tvF.pil_to_tensor(Image.open(mask_fname)).bool()
        
        try:
            next_rgb_fname = rgbs[i+interval]
            next_mask_fname = masks[i+interval]
        except:
            break
        
        next_rgb = tvF.pil_to_tensor(Image.open(next_rgb_fname).convert('L')).unsqueeze(0).float()/255
        next_mask = tvF.pil_to_tensor(Image.open(next_mask_fname)).bool()
        
        forward_dict = {
            'image0': rgb.to(device),
            'image1': next_rgb.to(device)
        }
        
        out = loftr(forward_dict)
        save_name = str(i).zfill(4) + '_' + str(i+interval).zfill(4) + '.pth'
        torch.save(out, save_path / save_name)
    # break

100%|██████████| 5/5 [02:19<00:00, 27.90s/it]


In [8]:
torch.save(out, 'test.pth')

In [9]:
load_out = torch.load('test.pth')

In [ ]:
type(load_out)

In [ ]:
load_out.keys()

In [4]:
out.keys()


dict_keys(['keypoints0', 'keypoints1', 'confidence', 'batch_indexes'])

In [6]:
out['confidence'].shape

torch.Size([81])

In [7]:
a = torch.zeros(800, 800).to(out['confidence'])

In [8]:
match_idx = out['keypoints0'].long()

In [9]:
a[match_idx[:, 1], match_idx[:, 0]] = out['confidence']

In [10]:
a

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')